In [ ]:
import numpy as np
import pandas as pd
import psycopg2
from tqdm import tqdm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [ ]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [ ]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [ ]:
EPOCHS = 1000
BATCH_SIZE = 32
VALID_SPLIT = 0.1

In [ ]:
LOOKBACK = 3
SEASONAL_PERIODS = (24, 24*7)
COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [ ]:
account_ids = ['05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '4f1c8b04-eb4b-4887-8c69-a096f0fda3ab', 'ad942d1a-15ab-4a0d-83a8-ae82183ece53']

df_filtered = df[df['account_id'].isin(account_ids)]

In [ ]:
# id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))

In [ ]:
id_pairs

In [ ]:
weights_chan = {}
for account_id, sales_channel_id in tqdm(id_pairs):
    
    if sales_channel_id == 'ALL':
        cond = (sales['account_id'] == account_id) & \
               (sales['status'].notna())
    else:
        cond = (sales['account_id'] == account_id) & \
               (sales['sales_channel_id'] == sales_channel_id) & \
               (sales['status'].notna())

    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
    weights_chan[account_id] = {} if account_id not in weights_chan else weights_chan[account_id]
    
    df = df_client_mod['n_orders'].fillna(0)
    df = df.loc[df.index <= END_DATE]
    
    weights_chan[account_id][sales_channel_id] = df.median()

In [ ]:
weights_df = pd.DataFrame(weights_chan)

weights_df

In [ ]:
weights_df = weights_df.stack().reset_index().rename(columns={'level_0': 'sales_channel_id', 'level_1': 'account_id', 0: 'weights'}).sort_values(by=['account_id', 'sales_channel_id'])

weights_df

In [ ]:
weights_chan_df = weights_df.groupby('account_id')['weights'].apply(lambda x: x / x.sum()).fillna(0).reset_index().rename(columns={0: 'weights'}).drop('level_1', axis=1)

weights_chan_df = pd.concat([weights_df['sales_channel_id'].reset_index(drop=True), weights_chan_df], axis=1)

weights_chan_df

In [ ]:
weights_acc_df = weights_df.groupby('account_id')['weights'].sum() / weights_df.groupby('account_id')['weights'].sum().sum()

weights_acc_df

In [ ]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [ ]:
model = 'GradientBoosting'

In [ ]:
df_forecast_metrics_norm = {}
scores = {}
df_forecast_metrics_norm[model] = {}

for account_id, sales_channel_id in id_pairs:
    
    print(account_id, sales_channel_id)
    
    # if sales_channel_id == 'ALL':
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['status'].notna())
    # else:
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['sales_channel_id'] == sales_channel_id) & \
    #            (sales['status'].notna())
    cond = (sales['account_id'] == account_id) & \
           (sales['sales_channel_id'] == sales_channel_id) & \
           (sales['status'].notna())

    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
    df_forecast_metrics_norm[model][account_id] = {} if account_id not in df_forecast_metrics_norm[model] else df_forecast_metrics_norm[model][account_id]
    for dataset in ['sales', 'orders']:
        if dataset == 'sales':
            col = 'price_total_agg'
        else:
            col = 'n_orders'
        
        df = df_client_mod[col].fillna(0)
        
        y_test = df.loc[(df.index >= START_DATE) & (df.index <= END_DATE)].copy()
    
        cursor.execute(f"select {dataset}_high, {dataset}_low, {dataset}_mean from public.forecast where account_id = '{account_id}' and channel = '{sales_channel_id}' and model = '{model}'")
        res = cursor.fetchall()
        
        df_forecast = pd.DataFrame(res, columns=[f'{dataset}_high', f'{dataset}_low', f'{dataset}_mean'])[:9].set_index(y_test.index)
        
        display(df_forecast)
        
        forecast_mean = df_forecast[f'{dataset}_mean']
        forecast_low = df_forecast[f'{dataset}_low']
        forecast_high = df_forecast[f'{dataset}_high']
        
        forecast = pd.concat([y_test, forecast_mean, forecast_low, forecast_high], axis=1)
        forecast.columns = ['actual', 'forecast', 'lower', 'upper']
        
        display(forecast)
        
        df_forecast = forecast.assign(
            covered_pts=lambda x:
                4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
                2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
                1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
            covered_width=lambda x: x['upper'] - x['lower'],
        )
        
        display(df_forecast)
        
        df_forecast_metrics = {}
        df_forecast_metrics['total_covered'] = df_forecast['covered_pts'].sum()
        df_forecast_metrics['avg_covered'] = df_forecast['covered_pts'].mean()
        df_forecast_metrics['avg_covered_width'] = df_forecast['covered_width'].mean()
        
        display(df_forecast_metrics)
        
        if dataset == 'orders':
            df_forecast_orders_metrics_norm = df_forecast_metrics['avg_covered']/(1 + np.log(1 + df_forecast_metrics['avg_covered_width']))
            print('orders_metric:', df_forecast_orders_metrics_norm)
        else: 
            df_forecast_sales_metrics_norm = df_forecast_metrics['avg_covered']/(1 + np.log(1 + df_forecast_metrics['avg_covered_width']))
            print('sales_metric:', df_forecast_sales_metrics_norm)
    
    weight_chan = weights_chan_df.loc[(weights_chan_df['account_id'] == account_id) & (weights_chan_df['sales_channel_id'] == sales_channel_id), 'weights'].values[0]
    print(weight_chan)
    df_forecast_metrics_norm[model][account_id][sales_channel_id] = weight_chan * (0.5*df_forecast_sales_metrics_norm + 0.5*df_forecast_orders_metrics_norm)
    print(df_forecast_metrics_norm[model][account_id][sales_channel_id])

In [ ]:
0.5862068965517241*(0.5*0.9405500038466145 + 0.5*0.2901697111945687)

In [ ]:
scores[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics_norm[model][acc_id].values())) for acc_id in df_forecast_metrics_norm[model].keys()])

In [ ]:
scores

In [ ]:
df_forecast_metrics_norm